In [21]:
import polars as pl
from importlib import reload
import nbformat

try:
    import plotly.io as pio
    pio.renderers.default = 'notebook_connected'
except ModuleNotFoundError:
    pio = None
    print('plotly is not installed; plots will require `pip install plotly`.')

import config_tbank_dataset as config
reload(config)
from config_tbank_dataset import *

import data_loader_tbank_dataset
reload(data_loader_tbank_dataset)
from data_loader_tbank_dataset import *

from polars_bridge import ensure_pandas_frame

import strategy
reload(strategy)
from strategy import *

import execution_backtester
reload(execution_backtester)

import plotting
reload(plotting)
from plotting import *

import portfolio
reload(portfolio)
from portfolio import *

import rolling_metrics
reload(rolling_metrics)
from rolling_metrics import *

from regime_filters import CombinedRegimeFilter, DrawdownFilter, VolatilityFilter, SpxVixFilter


In [22]:
print_config_settings()

=== CONFIG SETTINGS ===
DATASET_DIR: /Users/vgshabanov/work/tbank-trader/data_exports/ru_shares_2022_daily
# Основные переменные
START_DATE: 2022-01-01
BACKTEST_START: 2022-01-01
END_DATE: 2026-04-19
INITIAL_INVESTMENT: 1000000
BENCHMARK_TICKERS_LIST: ['SBER', 'LKOH', 'GAZP']
# Strategy parameters
REBALANCE_FREQ: W
MOMENTUM_PERIODS: [30, 90, 126]
MOMENTUM_RANK: 95
CASH_RETURN_RATE: 0.0
# Data quality / universe controls
CALENDAR_WEEKDAYS_ONLY: True
CALENDAR_MIN_ACTIVE_COVERAGE_RATIO: 0.75
UNIVERSE_MIN_HISTORY_DAYS: 126
LIQUIDITY_WINDOW_DAYS: 63
MIN_MEDIAN_TURNOVER_RUB: 20000000.0
MAX_DAILY_RETURN: 1.0
MIN_DAILY_RETURN: -0.5
# Transaction cost parameters
TC_FIXED: 5.0
TC_PCT: 0.0004
LOT_SIZE: 1
MANAGEMENT_FEE_MONTHLY: 0.0

# =============================================================================

# REGIME FILTERS CONFIGURATION

# =============================================================================
# SPX/VIX filter is intentionally disabled for the local MOEX dataset noteb

## Выгрузка данных и формирование весов

Этот notebook повторяет логику `Main_SPX_momentum_weekly.ipynb`, но работает на локальном MOEX dataset из `tbank-trader/data_exports/ru_shares_2022_daily`.

Что меняется по сравнению с SPX-версией:
- `price_table_wo_divs` = **raw daily open** для исполнения
- `price_table_with_divs` = **dividend-adjusted daily close** для ранжирования
- `lot_sizes` берутся по тикерам из локального датасета
- сплиты поддерживаются через optional `splits.csv`, если он появится рядом с dataset
- режимный фильтр по умолчанию локальный drawdown-based, без `SPX/VIX`


In [23]:
raw_tickers = load_tickers()
lot_sizes_full = load_lot_sizes(raw_tickers)

price_table_wo_divs_raw, price_table_with_divs_raw, divs_raw = fetch_price_data_divs(
    raw_tickers,
    start=START_DATE,
    end=END_DATE,
)
raw_close_prices_raw = get_raw_close_prices(
    raw_tickers,
    start=START_DATE,
    end=END_DATE,
)
turnover_table_raw = get_turnover_table(
    raw_tickers,
    start=START_DATE,
    end=END_DATE,
)
index_price_table_raw = get_benchmark_prices(
    BENCHMARK_TICKERS_LIST,
    start=START_DATE,
    end=END_DATE,
)
status_snapshot = load_status_snapshot()

valid_trading_dates, calendar_report = build_market_calendar(
    price_table_wo_divs_raw,
    min_active_coverage_ratio=CALENDAR_MIN_ACTIVE_COVERAGE_RATIO,
    weekdays_only=CALENDAR_WEEKDAYS_ONLY,
)

price_table_wo_divs = filter_frame_to_dates(price_table_wo_divs_raw, valid_trading_dates)
price_table_with_divs = filter_frame_to_dates(price_table_with_divs_raw, valid_trading_dates)
divs = filter_frame_to_dates(divs_raw, valid_trading_dates)
raw_close_prices = filter_frame_to_dates(raw_close_prices_raw, valid_trading_dates)
turnover_table = filter_frame_to_dates(turnover_table_raw, valid_trading_dates)
index_price_table = filter_frame_to_dates(index_price_table_raw, valid_trading_dates)

eligible_tickers, eligibility_report = build_universe_eligibility(
    price_table=price_table_wo_divs,
    turnover_table=turnover_table,
    min_history_days=UNIVERSE_MIN_HISTORY_DAYS,
    liquidity_window=LIQUIDITY_WINDOW_DAYS,
    min_median_turnover_rub=MIN_MEDIAN_TURNOVER_RUB,
    max_daily_return=MAX_DAILY_RETURN,
    min_daily_return=MIN_DAILY_RETURN,
)

tickers = eligible_tickers
lot_sizes = {ticker: lot_sizes_full[ticker] for ticker in tickers if ticker in lot_sizes_full}

price_table_wo_divs = select_tickers(price_table_wo_divs, tickers)
price_table_with_divs = select_tickers(price_table_with_divs, tickers)
divs = select_tickers(divs, tickers)
raw_close_prices = select_tickers(raw_close_prices, tickers)
turnover_table = select_tickers(turnover_table, tickers)

dataset_summary = {
    'raw_ticker_count': len(raw_tickers),
    'eligible_ticker_count': len(tickers),
    'raw_date_count': price_table_wo_divs_raw.height,
    'valid_date_count': price_table_wo_divs.height,
    'dropped_date_count': price_table_wo_divs_raw.height - price_table_wo_divs.height,
    'corp_action_exclusions': eligibility_report.filter(~pl.col('passes_corp_action_screen')).height,
    'liquidity_exclusions': eligibility_report.filter(~pl.col('passes_liquidity')).height,
    'history_exclusions': eligibility_report.filter(~pl.col('passes_history')).height,
    'benchmark_shape': index_price_table.shape,
    'date_min': str(price_table_wo_divs.select(pl.col('date').min()).item()),
    'date_max': str(price_table_wo_divs.select(pl.col('date').max()).item()),
}
display(pl.DataFrame([dataset_summary]))
display(calendar_report.filter(~pl.col('keep')).head(10))
display(eligibility_report.filter(~pl.col('eligible')).head(20))


raw_ticker_count,eligible_ticker_count,raw_date_count,valid_date_count,dropped_date_count,corp_action_exclusions,liquidity_exclusions,history_exclusions,benchmark_shape,date_min,date_max
i64,i64,i64,i64,i64,i64,i64,i64,list[i64],str,str
155,64,1253,1066,187,1,88,4,"[1066, 4]","""2022-01-03 00:00:00""","""2026-04-17 00:00:00"""


date,present_count,active_count,coverage_ratio,is_weekday,keep
datetime[μs],i64,i64,f64,bool,bool
2022-01-07 00:00:00,1,128,0.0078125,true,false
2022-01-15 00:00:00,1,128,0.0078125,false,false
2022-01-22 00:00:00,1,128,0.0078125,false,false
2022-01-29 00:00:00,1,128,0.0078125,false,false
2022-02-05 00:00:00,1,128,0.0078125,false,false
2022-02-12 00:00:00,1,128,0.0078125,false,false
2022-02-19 00:00:00,1,128,0.0078125,false,false
2022-02-23 00:00:00,1,128,0.0078125,true,false
2022-02-26 00:00:00,1,128,0.0078125,false,false


ticker,history_obs,max_observed_return,min_observed_return,recent_median_turnover_rub,passes_history,passes_corp_action_screen,passes_liquidity,eligible
str,i64,f64,f64,f64,bool,bool,bool,bool
"""ABIO""",1066,0.31797,-0.238095,681376.5,true,true,false,false
"""ABRD""",1066,0.23946,-0.125773,289482.6,true,true,false,false
"""AFKS""",1066,0.139249,-0.249687,7.1145e6,true,true,false,false
"""AKRN""",1066,0.399768,-0.162257,1.7645166e7,true,true,false,false
"""APTK""",1066,0.216514,-0.222786,689016.6,true,true,false,false
…,…,…,…,…,…,…,…,…
"""DELI""",555,0.140097,-0.10344,1.0385e7,true,true,false,false
"""DOMRF""",103,0.040676,-0.022308,4.61055936e8,false,true,true,false
"""DVEC""",1066,0.355882,-0.195214,2172.64,true,true,false,false


In [24]:
# Рассчет доходностей по периодам (multi-period momentum)
momentum_returns = momentum_calculate_returns(
    price_table_with_divs,
    periods=MOMENTUM_PERIODS,
)
print(f"Momentum periods: {MOMENTUM_PERIODS}")

# Нахождение дат ребалансировок
momentum_rebalance_dates = momentum_get_rebalance_dates(
    momentum_returns,
    freq=REBALANCE_FREQ,
    momentum_period=MOMENTUM_PERIODS[0],
)

# HQM rank scores and target weights
rank_score_dict = momentum_rank_score_calc(momentum_rebalance_dates, momentum_returns)
hqm_table = momentum_hqm_table_calc(rank_score_dict, momentum_returns, rank=MOMENTUM_RANK)
momentum_weights = momentum_weights_table_calc(hqm_table, price_table_with_divs, freq=REBALANCE_FREQ)

print(f"Rebalance dates: {len(momentum_rebalance_dates)}")
print(f"First rebalance date: {momentum_rebalance_dates[0]}")
print(f"Last rebalance date: {momentum_rebalance_dates[-1]}")


Momentum periods: [30, 90, 126]
Rebalance dates: 220
First rebalance date: 2022-01-03 00:00:00
Last rebalance date: 2026-04-13 00:00:00


In [25]:
# Веса для дат ребалансировок
target_weights_rebal = momentum_weights.filter(pl.col('date').is_in(momentum_rebalance_dates))
target_weights_rebal.head(5)


date,AFLT,ALRS,AQUA,ASTR,BANE,BANEP,CHMF,CNRU,DATA,DIAS,ENPG,ETLN,EUTR,FLOT,GAZP,GMKN,HEAD,IVAT,LEAS,LENT,LKOH,LSRG,MAGN,MBNK,MDMG,MGNT,MOEX,MTLR,MTSS,MVID,NLMK,NVTK,OZON,PHOR,PIKK,PLZL,POSI,PRMD,RAGR,RNFT,ROSN,RTKM,RUAL,SBER,SBERP,SELG,SFIN,SIBN,SMLT,SNGSP,SPBE,SVAV,T,TATN,TATNP,TRNFP,UWGN,VKCO,VSEH,VSMO,VTBR,WUSH,X5,YDEX
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2022-01-03 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-10 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-17 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-24 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-31 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Backtest с учетом фильтров

По умолчанию здесь отключен `SPX/VIX`, потому что notebook должен работать только на локальном MOEX dataset.
Зато можно включать локальные drawdown / volatility filters, не меняя основной execution-based pipeline.


In [26]:
combined_filter = CombinedRegimeFilter(logic=REGIME_FILTER_LOGIC)

if REGIME_SPX_VIX_ENABLED:
    combined_filter.add_filter(SpxVixFilter(
        enabled=True,
        spx_ma_period=SPX_MA_PERIOD,
        vix_threshold=VIX_THRESHOLD,
        spx_ticker=SPX_TICKER,
        vix_ticker=VIX_TICKER,
    ))

if REGIME_DRAWDOWN_ENABLED:
    combined_filter.add_filter(DrawdownFilter(
        enabled=True,
        window=DRAWDOWN_WINDOW,
        threshold=DRAWDOWN_THRESHOLD,
        source=DRAWDOWN_SOURCE,
    ))

if REGIME_VOLATILITY_ENABLED:
    combined_filter.add_filter(VolatilityFilter(
        enabled=True,
        window=VOLATILITY_WINDOW,
        threshold=VOLATILITY_THRESHOLD,
        source=VOLATILITY_SOURCE,
    ))

combined_filter.get_enabled_filter_names()


['DRAWDOWN']

In [27]:
prices_df_exec = ensure_pandas_frame(price_table_wo_divs)
target_weights_exec = ensure_pandas_frame(target_weights_rebal)
benchmark_prices_for_filters = ensure_pandas_frame(index_price_table)

(equity_filtered, returns_filtered, snapshots_filtered, trades_filtered) = execution_backtester.run_execution_backtest_with_filters(
    prices_df=prices_df_exec,
    benchmark_prices_df=benchmark_prices_for_filters,
    target_weights_df=target_weights_exec,
    rebalance_dates=momentum_rebalance_dates,
    initial_capital=INITIAL_INVESTMENT,
    lot_size=lot_sizes,
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
    cash_return_rate=CASH_RETURN_RATE,
    apply_cash_yield=(CASH_RETURN_RATE > 0),
    start_date=BACKTEST_START,
    management_fee_monthly=MANAGEMENT_FEE_MONTHLY,
    regime_filter=combined_filter,
)


Running execution backtest with dynamic filters...
  Start date: 2022-01-03
  End date: 2026-04-17
  Trading days: 1066
  Rebalance dates: 220
  Initial capital: $1,000,000
  Active filters: ['DRAWDOWN']
  Filter logic: any
  Processed 100/1066 days, equity: $1,000,000, regime: ON
  📉 2022-09-21: Regime OFF - liquidated 3 positions
  Processed 200/1066 days, equity: $982,092, regime: ON
  Processed 300/1066 days, equity: $1,846,939, regime: ON
  📉 2023-05-04: Regime OFF - liquidated 3 positions
  Processed 400/1066 days, equity: $2,302,664, regime: ON
  📉 2023-09-20: Regime OFF - liquidated 3 positions
  Processed 500/1066 days, equity: $2,321,285, regime: ON
  📉 2024-06-03: Regime OFF - liquidated 3 positions
  📉 2024-06-11: Regime OFF - liquidated 3 positions
  Processed 600/1066 days, equity: $2,979,541, regime: OFF
  Processed 700/1066 days, equity: $2,308,298, regime: ON
  Processed 800/1066 days, equity: $2,097,996, regime: OFF
  Processed 900/1066 days, equity: $2,125,835, regim

In [28]:
display(equity_filtered.head(5).to_frame('equity'))
display(returns_filtered.head(5).to_frame('returns'))
display(snapshots_filtered.head(5))
display(trades_filtered.head(20))

,equity
date,
2022-01-03,1000000.0
2022-01-04,1000000.0
2022-01-05,1000000.0
2022-01-06,1000000.0
2022-01-10,1000000.0


,returns
date,
2022-01-03,0.0
2022-01-04,0.0
2022-01-05,0.0
2022-01-06,0.0
2022-01-10,0.0


,equity,cash,holdings_value,num_positions,costs_today,mgmt_fee_today,regime_on,rolling_dd_30d,rolling_vol_30d,filter_DRAWDOWN_on,filter_DRAWDOWN_value
date,,,,,,,,,,,
2022-01-03,1000000.0,1000000.0,0.0,0,0.0,0.0,True,NaN,NaN,NaN,NaN
2022-01-04,1000000.0,1000000.0,0.0,0,0.0,0.0,True,0.0,NaN,True,0.0
2022-01-05,1000000.0,1000000.0,0.0,0,0.0,0.0,True,0.0,0.0,True,0.0
2022-01-06,1000000.0,1000000.0,0.0,0,0.0,0.0,True,0.0,0.0,True,0.0
2022-01-10,1000000.0,1000000.0,0.0,0,0.0,0.0,True,0.0,0.0,True,0.0


,date,ticker,side,shares,price,notional,cost,reason
0,2022-08-08,POSI,BUY,336,1015.20,341107.20,141.44288,REBALANCE
1,2022-08-08,MTLR,BUY,2755,120.99,333327.45,138.33098,REBALANCE
2,2022-08-08,MTSS,BUY,1320,245.00,323400.00,134.36000,REBALANCE
3,2022-08-15,MTLR,SELL,6,131.60,789.60,5.31584,REBALANCE
4,2022-08-15,MTSS,SELL,1320,247.70,326964.00,135.78560,REBALANCE
5,2022-08-15,POSI,SELL,336,1235.00,414960.00,170.98400,REBALANCE
6,2022-08-15,MGNT,BUY,72,5150.00,370800.00,153.32000,REBALANCE
7,2022-08-15,SVAV,BUY,1876,196.50,368634.00,152.45360,REBALANCE
8,2022-08-22,MGNT,SELL,1,5243.00,5243.00,7.09720,REBALANCE
9,2022-08-22,MTLR,SELL,2749,125.70,345549.30,143.21972,REBALANCE


In [29]:
fig = plot_strategy_equity(
    strategy_equity=equity_filtered,
    rebalance_dates=momentum_rebalance_dates,
    title='MOEX Momentum Equity Curve (Filtered)',
    snapshots_df=snapshots_filtered,
)
fig.show()

expanding_df = calculate_expanding_metrics(equity_filtered)
latest_date = expanding_df.index[-1]

metrics_table = pl.DataFrame({
    'latest_date': [latest_date],
    'equity': [equity_filtered.iloc[-1]],
    'cagr': [expanding_df['cagr'].iloc[-1] if not expanding_df.empty else None],
    'vol': [expanding_df['vol'].iloc[-1] if not expanding_df.empty else None],
    'sharpe': [expanding_df['sharpe'].iloc[-1] if not expanding_df.empty else None],
    'mdd': [expanding_df['mdd'].iloc[-1] if not expanding_df.empty else None],
})
metrics_table


latest_date,equity,cagr,vol,sharpe,mdd
datetime[μs],f64,f64,f64,f64,f64
2026-04-17 00:00:00,1.4908e6,0.099087,0.317211,0.455439,-0.583873


## Сравнение с локальными бенчмарками

Вместо Yahoo-бенчмарков здесь используются локальные бумаги из того же датасета.
По умолчанию это `SBER / LKOH / GAZP`, но список можно менять в `config_tbank_dataset.py`.


In [30]:
benchmark_lot_sizes = load_lot_sizes(BENCHMARK_TICKERS_LIST)
benchmark_prices_exec = ensure_pandas_frame(index_price_table)
benchmark_results = execution_backtester.run_benchmark_backtest(
    benchmark_tickers=BENCHMARK_TICKERS_LIST,
    prices_df=benchmark_prices_exec,
    initial_capital=INITIAL_INVESTMENT,
    lot_size=benchmark_lot_sizes,
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
    cash_return_rate=CASH_RETURN_RATE,
    apply_cash_yield=(CASH_RETURN_RATE > 0),
    start_date=BACKTEST_START,
)

fig = plot_strategy_vs_benchmarks(
    strategy_equity=equity_filtered,
    benchmark_results=benchmark_results,
    rebalance_dates=momentum_rebalance_dates,
    title='MOEX Momentum vs Local Benchmarks',
)
fig.show()



Running buy-and-hold benchmarks...
  Benchmarks: ['SBER', 'LKOH', 'GAZP']
  Initial capital: $1,000,000
  Transaction costs: $5.0 + 4.0 bps
  ✓ SBER: 3272 shares @ $305.47, Final: $1,455,019.77 (45.50%)
  ✓ LKOH: 149 shares @ $6683.00, Final: $1,378,643.55 (37.86%)
  ✓ GAZP: 2820 shares @ $353.73, Final: $463,823.10 (-53.62%)


## Расчет позиций для клиентского портфеля

Секция использует те же target weights, но считает реальные позиции с учетом российских lot sizes.


In [31]:
CLIENT_PORTFOLIO_SIZE = 10000
PORTFOLIO_DATE = None
SAVE_TO_EXCEL = False

positions_df, summary, target_date, output_path = build_client_portfolio_positions(
    weights_df=ensure_pandas_frame(target_weights_rebal),
    prices_df=ensure_pandas_frame(raw_close_prices),
    portfolio_size=CLIENT_PORTFOLIO_SIZE,
    portfolio_date=PORTFOLIO_DATE,
    lot_size=lot_sizes,
    rounding_rule='floor',
    save_to_excel=SAVE_TO_EXCEL,
    output_dir='data',
    verbose=True,
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
)
positions_df.head(20)


=== Client Portfolio Calculation ===
Target date: 2026-04-13
Portfolio size: $10,000

--- Position Summary ---
Number of positions: 4
Total invested (before costs): $8,692.28
Cash remaining (before costs): $1,307.72
Actual weight sum: 86.92%

--- Estimated Transaction Costs ---
Fixed costs (4 orders × $5.0): $20.00
Proportional costs (0.04% × $8,692): $3.48
Total estimated costs: $23.48

--- Net Summary ---
Net invested (with costs): $8,715.76
Net cash remaining: $1,284.24

List of positions:


,ticker,target_weight,price,shares,position_value,actual_weight
0,RUAL,0.256198,43.110,50,2155.500,0.215550
1,VTBR,0.252066,91.955,27,2482.785,0.248278
2,SFIN,0.247934,970.000,2,1940.000,0.194000
3,PIKK,0.243802,528.500,4,2114.000,0.211400


## Сравнение стратегии с фильтрами / без фильтров


In [32]:
equity_curve_exec_no_filters, daily_returns_exec_no_filters, snapshots_df_no_filters, trades_df_no_filters = execution_backtester.run_execution_backtest(
    prices_df=prices_df_exec,
    target_weights_df=target_weights_exec,
    rebalance_dates=momentum_rebalance_dates,
    initial_capital=INITIAL_INVESTMENT,
    lot_size=lot_sizes,
    tc_fixed=TC_FIXED,
    tc_pct=TC_PCT,
    cash_return_rate=CASH_RETURN_RATE,
    apply_cash_yield=(CASH_RETURN_RATE > 0),
    start_date=BACKTEST_START,
    management_fee_monthly=MANAGEMENT_FEE_MONTHLY,
)

fig = plot_filtered_backtest(
    equity_unfiltered=equity_curve_exec_no_filters,
    equity_filtered=equity_filtered,
    snapshots_filtered=snapshots_filtered,
    rebalance_dates=momentum_rebalance_dates,
    drawdown_threshold=DRAWDOWN_THRESHOLD,
    initial_investment=INITIAL_INVESTMENT,
    title='MOEX Momentum: With vs Without Filters',
)
fig.show()


Running execution backtest...
  Start date: 2022-01-03
  End date: 2026-04-17
  Trading days: 1066
  Rebalance dates: 220
  Initial capital: $1,000,000
  Transaction costs: $5.0 + 4.0 bps
  Processed 100/1066 days, equity: $1,000,000
  Processed 200/1066 days, equity: $1,089,502
  Processed 300/1066 days, equity: $2,049,147
  Processed 400/1066 days, equity: $3,123,181
  Processed 500/1066 days, equity: $3,476,123
  Processed 600/1066 days, equity: $4,966,694
  Processed 700/1066 days, equity: $4,399,295
  Processed 800/1066 days, equity: $4,301,473
  Processed 900/1066 days, equity: $4,248,839
  Processed 1000/1066 days, equity: $3,740,955
  Processed 1066/1066 days, equity: $3,827,874

=== Backtest Complete ===
Final equity: $3,827,874.41
Total return: 282.79%
Total trades: 849
Total transaction costs: $189,374.64
Total management fees: $0.00
Total all costs: $189,374.64
Cash never went negative: ✓
